In [10]:
import os
print(os.getcwd())

c:\Users\mmira\OneDrive\Documents\Project\Crop-Yield-Prediction\corn-yield-prediction-team5\notebooks


In [12]:
import sys
import os

# Walks up from wherever the notebook is until it finds the project root
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

import pandas as pd
import numpy as np
from src.features import engineer_features


from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error

from xgboost import XGBRegressor

In [13]:
df = pd.read_csv("../data/raw/2022/DataPublication_final/GroundTruth/HYBRID_HIPS_V3.5_ALLPLOTS.csv")
df.head()

,index,qrCode,location,irrigationProvided,nitrogenTreatment,poundsOfNitrogenPerAcre,experiment,plotLength,block,row,range,plotNumber,genotype,plantingDate,totalStandCount,daysToAnthesis,GDDToAnthesis,yieldPerAcre
0,8,NaN,Ames,NaN,NaN,0,4231,NaN,NaN,2,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11,NaN,Ames,NaN,NaN,0,4231,NaN,NaN,2,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,NaN,Ames,NaN,NaN,0,4231,NaN,NaN,3,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9,NaN,Ames,NaN,NaN,0,4231,NaN,NaN,3,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12,NaN,Ames,NaN,NaN,0,4231,NaN,NaN,3,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print("Shape before cleaning:", df.shape)
print("Missing yield values:", df["yieldPerAcre"].isna().sum())

In [ ]:
df = df.dropna(subset=["yieldPerAcre"]).copy()

print("Shape after dropping missing target:", df.shape)

In [ ]:
columns_to_drop_if_present = [
    "plotNumber",
    "qrCode",
    "totalStandCount",
    "daysToAnthesis",
    "GDDToAnthesis"
]

existing_drop_cols = [col for col in columns_to_drop_if_present if col in df.columns]

df = df.drop(columns=existing_drop_cols)

print("Dropped columns:", existing_drop_cols)
print("Shape after column drop:", df.shape)

In [ ]:
target_col = "yieldPerAcre"

X = df.drop(columns=[target_col])
y = df[target_col]


X = engineer_features(X)

print("X shape after engineering:", X.shape)
print("X columns after engineering:\n", X.columns.tolist())

In [ ]:
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

high_cardinality = [
    col for col in categorical_features
    if X[col].nunique() > 50
]

print("High-cardinality columns:", high_cardinality)

X = X.drop(columns=high_cardinality)

print("New shape after dropping:", X.shape)

In [ ]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Number of numeric columns:", len(numeric_features))
print("Number of categorical columns:", len(categorical_features))
print("\nSample numeric columns:", numeric_features[:10])
print("\nSample categorical columns:", categorical_features[:10])

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

In [ ]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", max_categories=10))
])

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

In [ ]:
model = XGBRegressor(
    n_estimators=1200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42
)

In [ ]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
import numpy as np

gkf = GroupKFold(n_splits=5)
groups = df.loc[X.index, "location"]

scores = []

for train_idx, test_idx in gkf.split(X, y, groups=groups):
    pipeline.fit(X.iloc[train_idx], y.iloc[train_idx])
    preds = pipeline.predict(X.iloc[test_idx])
    rmse = np.sqrt(mean_squared_error(y.iloc[test_idx], preds))
    scores.append(rmse)

print("Group CV RMSE scores:", scores)
print("Mean Group CV RMSE:", np.mean(scores))
print("Std Group CV RMSE:", np.std(scores))

In [ ]:
pipeline.fit(X_train, y_train)
print("Training complete.")

In [ ]:
preds = pipeline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, preds))

print("RMSE:", rmse)

In [ ]:
results = pd.DataFrame({
    "actual": y_test.values,
    "predicted": preds
})

results.head(10)

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    pipeline,
    X,
    y,
    scoring="neg_mean_squared_error",
    cv=5
)

rmse_scores = np.sqrt(-scores)

print("CV RMSE scores:", rmse_scores)
print("Mean CV RMSE:", rmse_scores.mean())

In [ ]:
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = pipeline.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

importance_df.head(20)

In [ ]:
import pickle

with open("../models/xgb_pipeline.pkl", "wb") as f:
    pickle.dump(pipeline, f)
